# Bài thực hành Deep Learning: Convolutional Neural Network (CNN)

**File notebook:** `CNN_Thuc_Hanh_DeepLearning.ipynb`

Notebook này thực hành xây dựng, huấn luyện, đánh giá và lưu các mô hình CNN bằng TensorFlow/Keras cho các bài toán nhận dạng ảnh phổ biến.

## 1. Tiêu đề bài thực hành

**Thực hành Convolutional Neural Network với TensorFlow/Keras**

Các bài toán trong notebook gồm:

- Nhận dạng ảnh CIFAR10.
- Nhận dạng ảnh Cat/Dog.
- Nhận dạng ảnh Fashion-MNIST.
- Nhận dạng khuôn mặt Nam/Nữ.
- Chuẩn bị hàm dự đoán ảnh mới để có thể tái sử dụng khi triển khai Flask.

## 2. Mục tiêu bài thực hành

Sau bài thực hành này, sinh viên có thể:

- Hiểu quy trình cơ bản khi xây dựng một mô hình CNN.
- Load dataset có sẵn hoặc dataset public.
- Tiền xử lý ảnh: resize, reshape, chuẩn hóa pixel về khoảng 0 đến 1.
- Xây dựng mô hình CNN gồm `Conv2D`, `MaxPooling2D`, `Flatten`, `Dense`, `Dropout`.
- Compile, train, đánh giá và vẽ biểu đồ accuracy/loss.
- Lưu mô hình `.h5` vào thư mục `models/`.
- Viết hàm dự đoán ảnh mới có thể dùng lại cho Flask.

## 3. Giới thiệu CNN

Convolutional Neural Network (CNN) là một kiến trúc mạng nơ-ron thường dùng cho dữ liệu ảnh. CNN học các đặc trưng không gian của ảnh thông qua các lớp tích chập.

Một mô hình CNN cơ bản thường gồm:

- **Conv2D:** trích xuất đặc trưng như cạnh, màu sắc, hình dạng.
- **MaxPooling2D:** giảm kích thước đặc trưng, giúp mô hình học nhanh hơn và giảm overfitting.
- **Flatten:** chuyển ma trận đặc trưng thành vector.
- **Dense:** phân loại dựa trên đặc trưng đã học.
- **Dropout:** giảm overfitting bằng cách bỏ ngẫu nhiên một phần neuron khi train.

Trong các bài toán phân loại nhiều lớp, ta thường dùng `softmax` và `sparse_categorical_crossentropy`. Với phân loại nhị phân như Cat/Dog hoặc Nam/Nữ, ta thường dùng `sigmoid` và `binary_crossentropy`.

## 4. Import thư viện

Cell dưới đây import các thư viện cần thiết, tạo thư mục `models/`, và định nghĩa các hàm dùng chung cho toàn bộ notebook.

In [ ]:
# Nếu chạy trên Google Colab và muốn lưu/đọc dữ liệu từ Google Drive,
# có thể bỏ comment các dòng dưới đây.
# from google.colab import drive
# drive.mount('/content/drive')

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print('TensorFlow version:', tf.__version__)

# Tạo thư mục lưu mô hình nếu chưa tồn tại
MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Cố định seed để kết quả dễ tái lập hơn
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
def build_cnn_model(input_shape, num_classes=2, binary=False):
    """Xây dựng mô hình CNN cơ bản dùng lại cho nhiều bài toán."""
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid') if binary else layers.Dense(num_classes, activation='softmax')
    ])

    if binary:
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    else:
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return model


def plot_training_history(history, title='Biểu đồ huấn luyện'):
    """Vẽ biểu đồ accuracy và loss sau khi train."""
    if history is None:
        print('Chưa có lịch sử huấn luyện để vẽ biểu đồ.')
        return

    acc = history.history.get('accuracy', [])
    val_acc = history.history.get('val_accuracy', [])
    loss = history.history.get('loss', [])
    val_loss = history.history.get('val_loss', [])

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(acc, label='Train Accuracy')
    if len(val_acc) > 0:
        plt.plot(val_acc, label='Validation Accuracy')
    plt.title(title + ' - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(loss, label='Train Loss')
    if len(val_loss) > 0:
        plt.plot(val_loss, label='Validation Loss')
    plt.title(title + ' - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()


def show_sample_images(images, labels, class_names, rows=2, cols=5, grayscale=False, title='Ảnh mẫu'):
    """Hiển thị một vài ảnh mẫu và nhãn tương ứng."""
    plt.figure(figsize=(cols * 2, rows * 2))
    for i in range(rows * cols):
        plt.subplot(rows, cols, i + 1)
        if grayscale:
            plt.imshow(images[i].squeeze(), cmap='gray')
        else:
            plt.imshow(images[i])
        label = int(np.array(labels[i]).squeeze())
        plt.title(class_names[label])
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def save_class_names(class_names, file_path):
    """Lưu danh sách nhãn để tiện dùng lại khi triển khai Flask."""
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(class_names, f, ensure_ascii=False, indent=2)

## 5. Bài 1: CNN nhận dạng CIFAR10

Dataset CIFAR10 gồm ảnh màu kích thước `32x32x3`, chia thành 10 lớp: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

In [ ]:
from tensorflow.keras.datasets import cifar10

cifar10_class_names = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# Load dataset CIFAR10 có sẵn trong Keras
(x_train_cifar, y_train_cifar), (x_test_cifar, y_test_cifar) = cifar10.load_data()

# Chuẩn hóa pixel từ [0, 255] về [0, 1]
x_train_cifar = x_train_cifar.astype('float32') / 255.0
x_test_cifar = x_test_cifar.astype('float32') / 255.0

print('Train shape:', x_train_cifar.shape)
print('Test shape:', x_test_cifar.shape)
show_sample_images(x_train_cifar, y_train_cifar, cifar10_class_names, title='Ảnh mẫu CIFAR10')

In [ ]:
cifar10_model = build_cnn_model(input_shape=(32, 32, 3), num_classes=10, binary=False)
cifar10_model.summary()

# Train khoảng 5 epoch để chạy nhanh trong môi trường thực hành
history_cifar10 = cifar10_model.fit(
    x_train_cifar, y_train_cifar,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

plot_training_history(history_cifar10, title='CIFAR10')

cifar10_test_loss, cifar10_test_acc = cifar10_model.evaluate(x_test_cifar, y_test_cifar, verbose=0)
print(f'Accuracy trên tập test CIFAR10: {cifar10_test_acc:.4f}')

cifar10_model_path = os.path.join(MODEL_DIR, 'cifar10_cnn.h5')
cifar10_model.save(cifar10_model_path)
save_class_names(cifar10_class_names, os.path.join(MODEL_DIR, 'cifar10_class_names.json'))
print('Đã lưu model:', cifar10_model_path)

## 6. Bài 2: CNN nhận dạng Cat/Dog

Bài này ưu tiên dùng `tensorflow_datasets` với dataset public `cats_vs_dogs`. Nếu không cài được `tensorflow_datasets` hoặc lỗi tải dataset, notebook sẽ chuyển sang hướng dẫn và code thay thế bằng `ImageDataGenerator` đọc dữ liệu từ thư mục:

```text
datasets/catdog/
  train/
    cat/
    dog/
  val/
    cat/
    dog/
```

In [ ]:
catdog_class_names = ['cat', 'dog']
catdog_model = None
history_catdog = None
catdog_eval_acc = None

IMG_SIZE_CATDOG = (128, 128)
BATCH_SIZE = 32

def preprocess_catdog_tfds(image_tensor, label):
    # Resize và chuẩn hóa ảnh về khoảng [0, 1]
    image_tensor = tf.image.resize(image_tensor, IMG_SIZE_CATDOG)
    image_tensor = tf.cast(image_tensor, tf.float32) / 255.0
    label = tf.cast(label, tf.float32)
    return image_tensor, label

try:
    import tensorflow_datasets as tfds

    print('Đang load cats_vs_dogs bằng tensorflow_datasets...')
    (catdog_train_ds, catdog_val_ds), catdog_info = tfds.load(
        'cats_vs_dogs',
        split=['train[:80%]', 'train[80%:]'],
        as_supervised=True,
        with_info=True
    )

    catdog_train_ds = (
        catdog_train_ds
        .map(preprocess_catdog_tfds, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(1000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )
    catdog_val_ds = (
        catdog_val_ds
        .map(preprocess_catdog_tfds, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    # Hiển thị một vài ảnh mẫu
    for batch_images, batch_labels in catdog_train_ds.take(1):
        show_sample_images(batch_images.numpy(), batch_labels.numpy(), catdog_class_names, title='Ảnh mẫu Cat/Dog')

    catdog_model = build_cnn_model(input_shape=(128, 128, 3), num_classes=2, binary=True)
    catdog_model.summary()

    history_catdog = catdog_model.fit(
        catdog_train_ds,
        validation_data=catdog_val_ds,
        epochs=5,
        verbose=1
    )

    plot_training_history(history_catdog, title='Cat/Dog')
    catdog_loss, catdog_eval_acc = catdog_model.evaluate(catdog_val_ds, verbose=0)
    print(f'Accuracy trên tập validation Cat/Dog: {catdog_eval_acc:.4f}')

except Exception as e:
    print('Không thể dùng tensorflow_datasets hoặc không tải được cats_vs_dogs.')
    print('Lỗi:', e)
    print('\nChuyển sang phương án ImageDataGenerator đọc dữ liệu local.')

    catdog_train_dir = os.path.join('datasets', 'catdog', 'train')
    catdog_val_dir = os.path.join('datasets', 'catdog', 'val')

    required_dirs = [
        os.path.join(catdog_train_dir, 'cat'),
        os.path.join(catdog_train_dir, 'dog'),
        os.path.join(catdog_val_dir, 'cat'),
        os.path.join(catdog_val_dir, 'dog')
    ]

    if not all(os.path.isdir(folder) for folder in required_dirs):
        print('\nChưa tìm thấy dataset Cat/Dog local.')
        print('Vui lòng đặt ảnh vào đúng cấu trúc thư mục:')
        print('datasets/catdog/train/cat')
        print('datasets/catdog/train/dog')
        print('datasets/catdog/val/cat')
        print('datasets/catdog/val/dog')
    else:
        train_datagen = ImageDataGenerator(rescale=1.0 / 255.0)
        val_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

        catdog_train_gen = train_datagen.flow_from_directory(
            catdog_train_dir,
            target_size=IMG_SIZE_CATDOG,
            batch_size=BATCH_SIZE,
            class_mode='binary'
        )
        catdog_val_gen = val_datagen.flow_from_directory(
            catdog_val_dir,
            target_size=IMG_SIZE_CATDOG,
            batch_size=BATCH_SIZE,
            class_mode='binary',
            shuffle=False
        )

        # Lấy tên lớp theo đúng class_indices của generator
        catdog_class_names = [None] * len(catdog_train_gen.class_indices)
        for class_name, class_id in catdog_train_gen.class_indices.items():
            catdog_class_names[class_id] = class_name
        print('Class indices:', catdog_train_gen.class_indices)

        sample_images, sample_labels = next(catdog_train_gen)
        show_sample_images(sample_images, sample_labels, catdog_class_names, title='Ảnh mẫu Cat/Dog local')

        catdog_model = build_cnn_model(input_shape=(128, 128, 3), num_classes=2, binary=True)
        catdog_model.summary()

        history_catdog = catdog_model.fit(
            catdog_train_gen,
            validation_data=catdog_val_gen,
            epochs=5,
            verbose=1
        )

        plot_training_history(history_catdog, title='Cat/Dog local')
        catdog_loss, catdog_eval_acc = catdog_model.evaluate(catdog_val_gen, verbose=0)
        print(f'Accuracy trên tập validation Cat/Dog: {catdog_eval_acc:.4f}')

if catdog_model is not None:
    catdog_model_path = os.path.join(MODEL_DIR, 'catdog_cnn.h5')
    catdog_model.save(catdog_model_path)
    save_class_names(catdog_class_names, os.path.join(MODEL_DIR, 'catdog_class_names.json'))
    print('Đã lưu model:', catdog_model_path)
else:
    print('Bỏ qua bước lưu model Cat/Dog vì chưa có dataset hoặc quá trình tải dataset thất bại.')

## 7. Bài 3: CNN nhận dạng Fashion-MNIST

Fashion-MNIST gồm ảnh xám kích thước `28x28x1`, chia thành 10 lớp: T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot.

In [ ]:
from tensorflow.keras.datasets import fashion_mnist

fashion_class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Load dataset Fashion-MNIST có sẵn trong Keras
(x_train_fashion, y_train_fashion), (x_test_fashion, y_test_fashion) = fashion_mnist.load_data()

# Chuẩn hóa pixel và thêm chiều channel cho ảnh grayscale
x_train_fashion = x_train_fashion.astype('float32') / 255.0
x_test_fashion = x_test_fashion.astype('float32') / 255.0
x_train_fashion = np.expand_dims(x_train_fashion, axis=-1)
x_test_fashion = np.expand_dims(x_test_fashion, axis=-1)

print('Train shape:', x_train_fashion.shape)
print('Test shape:', x_test_fashion.shape)
show_sample_images(x_train_fashion, y_train_fashion, fashion_class_names, grayscale=True, title='Ảnh mẫu Fashion-MNIST')

In [ ]:
fashion_model = build_cnn_model(input_shape=(28, 28, 1), num_classes=10, binary=False)
fashion_model.summary()

history_fashion = fashion_model.fit(
    x_train_fashion, y_train_fashion,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

plot_training_history(history_fashion, title='Fashion-MNIST')

fashion_test_loss, fashion_test_acc = fashion_model.evaluate(x_test_fashion, y_test_fashion, verbose=0)
print(f'Accuracy trên tập test Fashion-MNIST: {fashion_test_acc:.4f}')

fashion_model_path = os.path.join(MODEL_DIR, 'fashion_mnist_cnn.h5')
fashion_model.save(fashion_model_path)
save_class_names(fashion_class_names, os.path.join(MODEL_DIR, 'fashion_mnist_class_names.json'))
print('Đã lưu model:', fashion_model_path)

In [ ]:
# Hiển thị ảnh mẫu kèm nhãn dự đoán của mô hình Fashion-MNIST
num_samples = 10
pred_probs = fashion_model.predict(x_test_fashion[:num_samples])
pred_labels = np.argmax(pred_probs, axis=1)

plt.figure(figsize=(14, 3))
for i in range(num_samples):
    plt.subplot(1, num_samples, i + 1)
    plt.imshow(x_test_fashion[i].squeeze(), cmap='gray')
    true_name = fashion_class_names[int(y_test_fashion[i])]
    pred_name = fashion_class_names[int(pred_labels[i])]
    plt.title(f'Thật: {true_name}\nDự đoán: {pred_name}', fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 8. Bài 4: CNN nhận dạng Nam/Nữ

Do giáo viên không cung cấp dataset, bài này đọc dữ liệu từ thư mục local. Cần chuẩn bị dữ liệu theo cấu trúc:

```text
datasets/gender/
  train/
    male/
    female/
  val/
    male/
    female/
```

Nếu chưa có dataset, cell bên dưới sẽ không dừng notebook mà in hướng dẫn đặt dữ liệu.

In [ ]:
gender_model = None
history_gender = None
gender_eval_acc = None
gender_class_names = ['female', 'male']

IMG_SIZE_GENDER = (128, 128)
gender_train_dir = os.path.join('datasets', 'gender', 'train')
gender_val_dir = os.path.join('datasets', 'gender', 'val')

gender_required_dirs = [
    os.path.join(gender_train_dir, 'male'),
    os.path.join(gender_train_dir, 'female'),
    os.path.join(gender_val_dir, 'male'),
    os.path.join(gender_val_dir, 'female')
]

if not all(os.path.isdir(folder) for folder in gender_required_dirs):
    print('Vui lòng tải dataset Nam/Nữ public, sau đó đặt ảnh vào đúng thư mục male/female.')
    print('\nCấu trúc thư mục cần có:')
    print('datasets/gender/train/male')
    print('datasets/gender/train/female')
    print('datasets/gender/val/male')
    print('datasets/gender/val/female')
else:
    gender_train_datagen = ImageDataGenerator(rescale=1.0 / 255.0)
    gender_val_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

    gender_train_gen = gender_train_datagen.flow_from_directory(
        gender_train_dir,
        target_size=IMG_SIZE_GENDER,
        batch_size=BATCH_SIZE,
        class_mode='binary'
    )
    gender_val_gen = gender_val_datagen.flow_from_directory(
        gender_val_dir,
        target_size=IMG_SIZE_GENDER,
        batch_size=BATCH_SIZE,
        class_mode='binary',
        shuffle=False
    )

    # Lấy tên lớp theo đúng thứ tự class_id của generator
    gender_class_names = [None] * len(gender_train_gen.class_indices)
    for class_name, class_id in gender_train_gen.class_indices.items():
        gender_class_names[class_id] = class_name
    print('Class indices:', gender_train_gen.class_indices)

    sample_images, sample_labels = next(gender_train_gen)
    show_sample_images(sample_images, sample_labels, gender_class_names, title='Ảnh mẫu Nam/Nữ')

    gender_model = build_cnn_model(input_shape=(128, 128, 3), num_classes=2, binary=True)
    gender_model.summary()

    history_gender = gender_model.fit(
        gender_train_gen,
        validation_data=gender_val_gen,
        epochs=5,
        verbose=1
    )

    plot_training_history(history_gender, title='Nam/Nữ')
    gender_loss, gender_eval_acc = gender_model.evaluate(gender_val_gen, verbose=0)
    print(f'Accuracy trên tập validation Nam/Nữ: {gender_eval_acc:.4f}')

    gender_model_path = os.path.join(MODEL_DIR, 'gender_cnn.h5')
    gender_model.save(gender_model_path)
    save_class_names(gender_class_names, os.path.join(MODEL_DIR, 'gender_class_names.json'))
    print('Đã lưu model:', gender_model_path)

## 9. Hàm dự đoán ảnh mới

Hàm `predict_image` bên dưới có thể dùng lại khi triển khai Flask. Ý tưởng khi đưa vào Flask:

- Upload ảnh từ form web.
- Lưu ảnh tạm vào thư mục `uploads/`.
- Gọi `predict_image(model_path, image_path, target_size, class_names, grayscale=False)`.
- Trả kết quả dự đoán ra giao diện web.

In [ ]:
def predict_image(model_path, image_path, target_size, class_names, grayscale=False):
    """
    Dự đoán nhãn cho một ảnh mới.

    Tham số:
    - model_path: đường dẫn file model .h5
    - image_path: đường dẫn ảnh cần dự đoán
    - target_size: kích thước resize, ví dụ (128, 128)
    - class_names: danh sách tên lớp
    - grayscale: True nếu model nhận ảnh xám, False nếu model nhận ảnh màu
    """
    if not os.path.exists(model_path):
        print('Không tìm thấy model:', model_path)
        return None

    if not os.path.exists(image_path):
        print('Không tìm thấy ảnh:', image_path)
        return None

    color_mode = 'grayscale' if grayscale else 'rgb'
    model = load_model(model_path)

    # Đọc ảnh, resize và chuẩn hóa pixel
    img = image.load_img(image_path, target_size=target_size, color_mode=color_mode)
    img_array = image.img_to_array(img).astype('float32') / 255.0
    img_batch = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_batch)

    # Xử lý cả mô hình binary sigmoid và multi-class softmax
    if prediction.shape[-1] == 1:
        prob_class_1 = float(prediction[0][0])
        predicted_index = 1 if prob_class_1 >= 0.5 else 0
        confidence = prob_class_1 if predicted_index == 1 else 1.0 - prob_class_1
    else:
        predicted_index = int(np.argmax(prediction[0]))
        confidence = float(np.max(prediction[0]))

    predicted_label = class_names[predicted_index]
    print(f'Nhãn dự đoán: {predicted_label}')
    print(f'Độ tin cậy: {confidence:.4f}')

    plt.figure(figsize=(4, 4))
    if grayscale:
        plt.imshow(img_array.squeeze(), cmap='gray')
    else:
        plt.imshow(img_array)
    plt.title(f'Dự đoán: {predicted_label} ({confidence:.2%})')
    plt.axis('off')
    plt.show()

    return predicted_label, confidence


# Ví dụ sử dụng sau khi đã có ảnh test:
# predict_image('models/catdog_cnn.h5', 'test_images/cat1.jpg', (128, 128), ['cat', 'dog'])
# predict_image('models/fashion_mnist_cnn.h5', 'test_images/shirt.png', (28, 28), fashion_class_names, grayscale=True)

## 10. Lưu mô hình

Các model được lưu vào thư mục `models/`. Nếu thư mục này chưa tồn tại thì notebook đã tự tạo bằng `os.makedirs(MODEL_DIR, exist_ok=True)`.

Các file model dự kiến:

- `models/cifar10_cnn.h5`
- `models/catdog_cnn.h5`
- `models/fashion_mnist_cnn.h5`
- `models/gender_cnn.h5`

Với các bài cần dataset local nhưng chưa có dữ liệu, notebook sẽ bỏ qua lưu model tương ứng để tránh lỗi dừng chương trình.

In [ ]:
# Kiểm tra các model đã được lưu trong thư mục models/
print('Danh sách file trong thư mục models/:')
for file_name in sorted(os.listdir(MODEL_DIR)):
    print('-', os.path.join(MODEL_DIR, file_name))

## 11. Nhận xét kết quả

- CIFAR10 khó hơn MNIST vì ảnh màu và nhiều đối tượng.
- Fashion-MNIST dễ hơn CIFAR10 vì ảnh đơn giản hơn.
- Cat/Dog là bài phân loại nhị phân.
- Nam/Nữ phụ thuộc nhiều vào chất lượng dataset.

Ngoài ra, với số epoch nhỏ từ 5 đến 10, mô hình chạy nhanh nhưng độ chính xác có thể chưa cao. Nếu muốn cải thiện kết quả, có thể tăng epoch, dùng data augmentation, điều chỉnh kiến trúc CNN hoặc dùng transfer learning với các mô hình như MobileNetV2, VGG16, EfficientNet.

### Bảng tổng kết

| Tên bài | Dataset | Kích thước ảnh đầu vào | Số lớp | Loss function | File model đã lưu |
|---|---|---:|---:|---|---|
| CIFAR10 | `tensorflow.keras.datasets.cifar10` | `32x32x3` | 10 | `sparse_categorical_crossentropy` | `models/cifar10_cnn.h5` |
| Cat/Dog | `tensorflow_datasets: cats_vs_dogs` hoặc `datasets/catdog/` | `128x128x3` | 2 | `binary_crossentropy` | `models/catdog_cnn.h5` |
| Fashion-MNIST | `tensorflow.keras.datasets.fashion_mnist` | `28x28x1` | 10 | `sparse_categorical_crossentropy` | `models/fashion_mnist_cnn.h5` |
| Nam/Nữ | `datasets/gender/` | `128x128x3` | 2 | `binary_crossentropy` | `models/gender_cnn.h5` |

## 12. Kết luận

Qua bài thực hành, sinh viên đã biết cách xây dựng CNN, huấn luyện mô hình bằng TensorFlow/Keras, đánh giá kết quả và lưu mô hình để sử dụng cho ứng dụng thực tế.